# Day 048 — Exercise 5: FeatureEngineer Class

**What you'll build:** The `FeatureEngineer` class — a preprocessing pipeline that composes `encode_categoricals`, `prepare_features`, `fit_scaler`, `scale_features`, and `split_data` behind two methods:
- `fit_transform(df, test_size=0.2, random_state=42) -> dict` — encode, scale (fit on train), split
- `transform(df) -> pd.DataFrame` — apply fitted encoding and scaling to new data using the same scaler learned during `fit_transform`

**Why it matters:** The same preprocessing pipeline must be applied identically to training and production data. By storing the scaler and feature column order, `FeatureEngineer` guarantees that new data is transformed the same way as the training data — no leakage, no column mismatch.

## Provided: All Helper Functions

In [ ]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')


def make_regression_data(n: int = 200, seed: int = 42) -> pd.DataFrame:
    """Synthetic housing dataset with one categorical column (neighborhood)."""
    rng = np.random.default_rng(seed)
    area         = rng.uniform(500, 3000, n).round(0)
    bedrooms     = rng.integers(1, 6, n)
    age          = rng.uniform(0, 50, n).round(1)
    neighborhood = rng.choice(['downtown', 'suburb', 'rural'], n)
    price = (
        area * 150
        + bedrooms * 10_000
        - age * 1_000
        + np.where(neighborhood == 'downtown', 50_000, 0)
        + np.where(neighborhood == 'suburb',   20_000, 0)
        + rng.standard_normal(n) * 10_000
    ).round(-2)
    return pd.DataFrame({
        'area':         area.astype(int),
        'bedrooms':     bedrooms,
        'age':          age,
        'neighborhood': neighborhood,
        'price':        price.astype(int),
    })


def prepare_features(df: pd.DataFrame, target_col: str,
                     numeric_only: bool = True):
    """Return (X, y) separating features from target."""
    X = df.drop(columns=[target_col])
    if numeric_only:
        X = X.select_dtypes(include='number')
    y = df[target_col]
    return X, y


def split_data(X: pd.DataFrame, y: pd.Series,
               test_size: float = 0.2,
               random_state: int = 42) -> dict:
    """Wrap train_test_split, return a result dict."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    return {
        'X_train':    X_train,
        'X_test':     X_test,
        'y_train':    y_train,
        'y_test':     y_test,
        'n_train':    len(X_train),
        'n_test':     len(X_test),
        'n_features': X_train.shape[1],
    }


def encode_categoricals(df: pd.DataFrame,
                         cat_cols: list | None = None,
                         drop_first: bool = False) -> pd.DataFrame:
    """One-hot encode categorical columns with pd.get_dummies."""
    if cat_cols is None:
        cat_cols = df.select_dtypes(include='object').columns.tolist()
    if not cat_cols:
        return df.copy()
    encoded = pd.get_dummies(df, columns=cat_cols, drop_first=drop_first)
    # pandas 2.x returns bool dtype for dummy columns; convert to int
    bool_cols = encoded.select_dtypes(include='bool').columns.tolist()
    for c in bool_cols:
        encoded[c] = encoded[c].astype(int)
    return encoded


from sklearn.preprocessing import StandardScaler


def fit_scaler(X_train: pd.DataFrame) -> StandardScaler:
    """Fit a StandardScaler on training data only."""
    scaler = StandardScaler()
    scaler.fit(X_train)
    return scaler


def scale_features(scaler: StandardScaler,
                   X: pd.DataFrame) -> pd.DataFrame:
    """Transform X using a fitted scaler; return DataFrame with same columns."""
    scaled = scaler.transform(X)
    return pd.DataFrame(scaled, columns=X.columns, index=X.index)


from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score


def train_model(X_train: pd.DataFrame,
                y_train: pd.Series) -> LinearRegression:
    """Fit LinearRegression on training data."""
    model = LinearRegression()
    model.fit(X_train, y_train)
    return model


def evaluate_model(model: LinearRegression,
                   X_test: pd.DataFrame,
                   y_test: pd.Series) -> dict:
    """Return R², RMSE, n_test, and predictions array."""
    y_pred = model.predict(X_test)
    r2     = r2_score(y_test, y_pred)
    rmse   = float(np.sqrt(mean_squared_error(y_test, y_pred)))
    return {
        'r2':          round(float(r2), 4),
        'rmse':        round(rmse, 2),
        'n_test':      len(y_test),
        'predictions': y_pred,
    }

## Your Implementation

In [ ]:
class FeatureEngineer:
    """
    End-to-end preprocessing pipeline: encode → scale → split.

    Usage:
        fe    = FeatureEngineer(target_col='price')
        split = fe.fit_transform(df)
        X_new = fe.transform(new_df)
    """

    def __init__(self, target_col: str,
                 cat_cols: list | None = None,
                 scale: bool = True):
        # TODO: self.target_col = target_col
        # TODO: self.cat_cols   = cat_cols
        # TODO: self.scale      = scale
        # TODO: self._scaler         = None
        # TODO: self._feature_cols   = None
        pass

    def fit_transform(self, df: pd.DataFrame,
                      test_size: float = 0.2,
                      random_state: int = 42) -> dict:
        """
        Encode categoricals, fit scaler on train, split.
        Stores feature column order and fitted scaler for transform().
        Returns the same split dict as split_data().
        """
        # TODO: encoded = encode_categoricals(df, cat_cols=self.cat_cols)
        # TODO: X, y   = prepare_features(encoded, self.target_col, numeric_only=False)
        # TODO: self._feature_cols = X.columns.tolist()
        # TODO: if self.scale:
        #     self._scaler = fit_scaler(X)
        #     X = scale_features(self._scaler, X)
        # TODO: return split_data(X, y, test_size=test_size, random_state=random_state)
        pass

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Apply fitted encoding + scaling to new data.
        Uses reindex to handle unseen categories gracefully.
        """
        # TODO: encoded = encode_categoricals(df, cat_cols=self.cat_cols)
        # TODO: X       = encoded.reindex(columns=self._feature_cols, fill_value=0)
        # TODO: if self.scale and self._scaler is not None:
        #     X = scale_features(self._scaler, X)
        # TODO: return X
        pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    df = make_regression_data(100)

    # Check 1: class defined with fit_transform and transform
    try:
        assert 'FeatureEngineer' in globals()
        for m in ('fit_transform', 'transform'):
            assert hasattr(FeatureEngineer, m), f'missing method: {m}'
        passed += 1; print('\u2705 Check 1: FeatureEngineer has fit_transform and transform')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: fit_transform returns dict with split keys
    try:
        fe    = FeatureEngineer(target_col='price')
        split = fe.fit_transform(df)
        assert isinstance(split, dict), \
            f'fit_transform must return dict, got {type(split).__name__}'
        for k in ('X_train', 'X_test', 'y_train', 'y_test'):
            assert k in split, f'missing key: {k!r}'
        passed += 1; print(f'\u2705 Check 2: fit_transform returns split dict with X_train, X_test, y_train, y_test')
    except Exception as e:
        print(f'\u274c Check 2: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 3: n_train + n_test == len(df)
    try:
        total_rows = split['n_train'] + split['n_test']
        assert total_rows == len(df), \
            f'n_train + n_test = {total_rows}, expected {len(df)}'
        passed += 1; print(f'\u2705 Check 3: n_train={split["n_train"]} + n_test={split["n_test"]} = {len(df)}')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: target_col not in X_train columns
    try:
        assert 'price' not in split['X_train'].columns, \
            'price should not appear in X_train'
        passed += 1; print(f'\u2705 Check 4: target col excluded from X_train')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: transform returns DataFrame with same columns as X_train
    try:
        new_df  = make_regression_data(10, seed=99)
        X_new   = fe.transform(new_df.drop(columns=['price']))
        assert isinstance(X_new, pd.DataFrame), \
            f'transform must return DataFrame, got {type(X_new).__name__}'
        assert list(X_new.columns) == list(split['X_train'].columns), \
            'transform output columns must match X_train columns'
        passed += 1; print(f'\u2705 Check 5: transform returns {X_new.shape} matching X_train columns')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
class FeatureEngineer:
    """
    End-to-end preprocessing pipeline: encode → scale → split.

    Usage:
        fe    = FeatureEngineer(target_col='price')
        split = fe.fit_transform(df)
        X_new = fe.transform(new_df)
    """

    def __init__(self, target_col: str,
                 cat_cols: list | None = None,
                 scale: bool = True):
        self.target_col   = target_col
        self.cat_cols     = cat_cols
        self.scale        = scale
        self._scaler      = None
        self._feature_cols = None

    def fit_transform(self, df: pd.DataFrame,
                      test_size: float = 0.2,
                      random_state: int = 42) -> dict:
        """Encode, scale (fit on train), split. Return split dict."""
        encoded             = encode_categoricals(df, cat_cols=self.cat_cols)
        X, y                = prepare_features(encoded, self.target_col,
                                               numeric_only=False)
        self._feature_cols  = X.columns.tolist()
        if self.scale:
            self._scaler = fit_scaler(X)
            X            = scale_features(self._scaler, X)
        return split_data(X, y, test_size=test_size,
                          random_state=random_state)

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """Apply fitted encoding + scaling to new data."""
        encoded = encode_categoricals(df, cat_cols=self.cat_cols)
        X       = encoded.reindex(columns=self._feature_cols, fill_value=0)
        if self.scale and self._scaler is not None:
            X = scale_features(self._scaler, X)
        return X
```

</details>